# Task 3 — Kafka topics and event contract

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | `screenshots/stage2_manifest.json` |
| Commands | `bash scripts/init_kafka_topics.sh`; `bash scripts/capture_kafka_evidence.sh` |
| Topics | Metadata, nodes, edges, and errors. |
| Required fields | `schema_version`, `event_time`, `repo`, `commit_sha`, `run_id`, `file_id`, `file_path`. |
```

## Approach and reasoning

Commands: `bash scripts/init_kafka_topics.sh` and `bash scripts/capture_kafka_evidence.sh`. The Kafka contract separates metadata, graph nodes, graph edges, and parser errors so each downstream consumer can subscribe to only the stream it owns. Every event carries provenance fields that tie the message back to the repository, commit, run, and file.

## What this chapter proves

| Requirement | Evidence in this chapter |
|---|---|
| Topic creation | The executed output records the expected Kafka topic layout. |
| Event schema | The chapter lists the common provenance fields required on each event. |
| Captured samples | The executed output renders one real node, edge, metadata, and error message from the run. |

Raw captured samples: [`cpg.nodes`](../screenshots/kafka/sample_cpg_nodes.json), [`cpg.edges`](../screenshots/kafka/sample_cpg_edges.json), [`cpg.metadata`](../screenshots/kafka/sample_cpg_metadata.json), and [`cpg.errors`](../screenshots/kafka/sample_cpg_errors.json).


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'screenshots/stage2_manifest.json').exists())
manifest = json.loads((ROOT / 'screenshots/stage2_manifest.json').read_text())
topics = ['cpg.nodes', 'cpg.edges', 'cpg.metadata', 'cpg.errors']
print('topics:', ', '.join(topics))
print('node events:', manifest['metrics']['kafka']['node_events'])
print('edge events:', manifest['metrics']['kafka']['edge_events'])
print('metadata/error:', manifest['metrics']['kafka']['metadata_events'], '/', manifest['metrics']['kafka']['error_events'])
print('topic_details.txt contains all four:', all(t in (ROOT / 'screenshots/kafka/topic_details.txt').read_text() for t in topics))

def first_event(filename):
    first_line = (ROOT / 'screenshots/kafka' / filename).read_text().splitlines()[0]
    return json.loads(first_line)

sample_files = {
    'cpg.nodes': 'sample_cpg_nodes.json',
    'cpg.edges': 'sample_cpg_edges.json',
    'cpg.metadata': 'sample_cpg_metadata.json',
    'cpg.errors': 'sample_cpg_errors.json',
}
for topic, filename in sample_files.items():
    event = first_event(filename)
    print(f'\n{topic} sample:')
    print(json.dumps(event, sort_keys=True))
    assert event['schema_version'] == '1.0'
    assert event['event_time'].endswith('Z')


topics: cpg.nodes, cpg.edges, cpg.metadata, cpg.errors
node events: 133263
edge events: 38141
metadata/error: 138 / 1
topic_details.txt contains all four: True

cpg.nodes sample:
{"col_offset": null, "commit_sha": "41adfd0f9ee9ba3a6b4f719d5b551c5b19ae45e2", "end_col_offset": null, "end_lineno": null, "event_time": "2026-07-23T06:58:02.430Z", "file_id": "6c39568a6a11c430", "file_path": "src/datasets/__init__.py", "id": "57d761e9567d5b95", "lineno": null, "node_type": "Module", "op": "upsert", "properties": {}, "repo": "huggingface/datasets", "run_id": "8c3efe458b3b43eaa1399d7740848d3f", "schema_version": "1.0", "scope_path": "<module>"}

cpg.edges sample:
{"commit_sha": "41adfd0f9ee9ba3a6b4f719d5b551c5b19ae45e2", "edge_type": "CFG_NEXT", "event_time": "2026-07-23T06:58:11.913Z", "file_id": "97d05a67e514a1e5", "file_path": "src/datasets/builder.py", "id": "07474daf0400bc31", "op": "upsert", "properties": {"extractor": "cfg"}, "repo": "huggingface/datasets", "run_id": "8c3efe458b3b43eaa13

## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | The four explicit topics were created with the expected partition layout, and node/edge IDs were unique across the captured run. |
| Failed | Provenance could previously drift between the parser and evidence capture. |
| Resolution | Capture now compares every sampled event with the exact cloned commit SHA, and the manifest hashes the raw artifacts. |
```
